# 从零训练迷你英中翻译模型 (Seq2Seq)

**使用 `gather` 方法按真实长度提取 Encoder 隐藏状态，解决 Padding 污染问题**。


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import itertools

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

## 1. 动态生成 3840 条数据集

In [8]:
def build_dataset():
    animals = {"duck": "鸭子", "cat": "猫", "dog": "狗", "cow": "牛", "bird": "鸟", "horse": "马", "bear": "熊", "lion": "狮子"}
    colors = {"quiet": "安静", "red": "红色", "black": "黑色", "white": "白色", "blue": "蓝色", "green": "绿色", "yellow": "黄色", "brown": "棕色"}
    locations = {"next to": "旁边", "behind": "后面", "in front of": "前面", "on": "上面", "under": "下面", "near": "附近"}
    objects = {"door": "门", "tree": "树", "table": "桌子", "chair": "椅子", "box": "盒子", "window": "窗户", "car": "汽车", "apple": "苹果"}
    names = {"alice": "爱丽丝", "bob": "鲍勃", "charlie": "查理", "david": "大卫", "emma": "艾玛", "fiona": "菲奥娜", "george": "乔治", "henry": "亨利"}
    numbers = {1: ("a", "one", "一"), 2: ("two", "two", "两"), 3: ("three", "three", "三"), 4: ("four", "four", "四"), 5: ("five", "five", "五"), 6: ("six", "six", "六")}

    pattern1_all = []
    for num, color, animal, loc, obj in itertools.product(numbers.keys(), colors.keys(), animals.keys(), locations.keys(), objects.keys()):
        en_num = numbers[num][0]
        zh_num = numbers[num][2]
        if num == 1:
            en = f"there is {en_num} {color} {animal} {loc} the {obj}"
        else:
            en = f"there are {en_num} {color} {animal}s {loc} the {obj}"
        zh = f"{objects[obj]} {locations[loc]} 有 {zh_num} 只 {colors[color]} {animals[animal]}"
        pattern1_all.append((en, zh))

    pattern2_all = []
    for name, num, color, obj in itertools.product(names.keys(), numbers.keys(), colors.keys(), objects.keys()):
        en_num = numbers[num][1]
        zh_num = numbers[num][2]
        if num == 1:
            en = f"{name} has {en_num} {color} {obj}"
        else:
            en = f"{name} has {en_num} {color} {obj}s"
        zh = f"{names[name]} 有 {zh_num} 个 {colors[color]} {objects[obj]}"
        pattern2_all.append((en, zh))

    random.seed(42)
    random.shuffle(pattern1_all)
    random.shuffle(pattern2_all)
    selected = pattern1_all[:1920] + pattern2_all[:1920]
    random.shuffle(selected)
    return selected

raw_data = build_dataset()

# 8:1:1 划分
train_size = int(3840 * 0.8)
val_size = int(3840 * 0.1)
train_data = raw_data[:train_size]
val_data = raw_data[train_size:train_size+val_size]
test_data = raw_data[train_size+val_size:]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# 构建词表
en_vocab = {"<PAD>":0, "<BOS>":1, "<EOS>":2, "<UNK>":3}
zh_vocab = {"<PAD>":0, "<BOS>":1, "<EOS>":2, "<UNK>":3}
for en, zh in train_data:
    for word in en.split():
        if word not in en_vocab: en_vocab[word] = len(en_vocab)
    for word in zh.split():
        if word not in zh_vocab: zh_vocab[word] = len(zh_vocab)

en_idx2word = {v: k for k, v in en_vocab.items()}
zh_idx2word = {v: k for k, v in zh_vocab.items()}
print(f"EN Vocab Size: {len(en_vocab)}, ZH Vocab Size: {len(zh_vocab)}")


Train: 3072, Val: 384, Test: 384
EN Vocab Size: 73, ZH Vocab Size: 51


## 2. 定义 Dataset 与 DataLoader (附带真实长度)

In [9]:
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

class TranslationDataset(Dataset):
    def __init__(self, data_pairs, en_vocab, zh_vocab):
        self.data_pairs = data_pairs
        self.en_vocab = en_vocab
        self.zh_vocab = zh_vocab

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        en, zh = self.data_pairs[idx]
        en_indices = [self.en_vocab.get(w, UNK_IDX) for w in en.split()]
        zh_indices = [self.zh_vocab.get(w, UNK_IDX) for w in zh.split()]

        # 1. 英文反转
        eng_reversed = en_indices[::-1]
        # 2. 中文加 BOS 和 EOS
        chn_target = [BOS_IDX] + zh_indices + [EOS_IDX]

        return torch.tensor(eng_reversed, dtype=torch.long), torch.tensor(chn_target, dtype=torch.long)

def collate_fn(batch):
    eng_batch, chn_batch = zip(*batch)

    # 获取每个英文句子的真实长度
    eng_lengths = torch.tensor([len(seq) for seq in eng_batch], dtype=torch.long)

    # 右侧填充
    eng_padded = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=PAD_IDX)
    chn_padded = torch.nn.utils.rnn.pad_sequence(chn_batch, batch_first=True, padding_value=PAD_IDX)

    return eng_padded, chn_padded, eng_lengths

train_dataset = TranslationDataset(train_data, en_vocab, zh_vocab)
val_dataset = TranslationDataset(val_data, en_vocab, zh_vocab)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)


## 3. 模型定义 (Gather 取最后状态)

In [10]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)

    def forward(self, source, source_lengths):
        # Token ID -> 词向量 -> GRU 逐步读入, outputs: [B, S, hidden_dim]
        outputs, _ = self.gru(self.embedding(source))

        # Padding 后的最后一列不一定是真实句尾，因此按真实长度取最后一个有效状态。
        # 索引 = 长度 - 1, 先扩展成 gather 需要的形状 [B, 1, hidden_dim]
        last_indices = (source_lengths - 1).view(-1, 1, 1).to(source.device)
        last_indices = last_indices.expand(-1, 1, outputs.size(-1))

        # 沿时间维取出每个句子的最后有效状态 -> [B, hidden_dim]
        final_hidden = outputs.gather(1, last_indices).squeeze(1)

        # 加回 num_layers 维 -> [1, B, hidden_dim], 可直接作为解码器 GRU 的初始隐藏状态
        return outputs, final_hidden.unsqueeze(0)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size=64, hidden_size=128):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, source, source_lengths, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[0]
        target_len = target.shape[1]
        target_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, target_len, target_vocab_size).to(self.device)

        _, hidden = self.encoder(source, source_lengths)
        x = target[:, 0].unsqueeze(1)

        for t in range(1, target_len):
            prediction, hidden = self.decoder(x, hidden)
            outputs[:, t, :] = prediction
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            x = target[:, t].unsqueeze(1) if teacher_force else top1.unsqueeze(1)

        return outputs


## 4. 训练与推理函数

In [11]:
ENG_VOCAB_SIZE = len(en_vocab)
CHN_VOCAB_SIZE = len(zh_vocab)
EMBED_SIZE = 64
HIDDEN_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(ENG_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
decoder = Decoder(CHN_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, dataloader, optimizer, criterion, device, clip=1.0):
    model.train()
    epoch_loss = 0
    for source, target, source_lengths in dataloader:
        source = source.to(device)
        target = target.to(device)
        source_lengths = source_lengths.to(device)

        optimizer.zero_grad()
        outputs = model(source, source_lengths, target, teacher_forcing_ratio=0.5)

        outputs = outputs[:, 1:].contiguous().view(-1, outputs.shape[-1])
        target = target[:, 1:].contiguous().view(-1)

        loss = criterion(outputs, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for source, target, source_lengths in dataloader:
            source = source.to(device)
            target = target.to(device)
            source_lengths = source_lengths.to(device)

            outputs = model(source, source_lengths, target, teacher_forcing_ratio=0.0)
            outputs = outputs[:, 1:].contiguous().view(-1, outputs.shape[-1])
            target = target[:, 1:].contiguous().view(-1)

            loss = criterion(outputs, target)
            epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def translate(model, english_indices, device, max_len=20):
    model.eval()
    with torch.no_grad():
        eng_reversed = english_indices[::-1]
        eng_len = torch.tensor([len(eng_reversed)], dtype=torch.long).to(device)
        english_tensor = torch.tensor(eng_reversed, dtype=torch.long).unsqueeze(0).to(device)

        _, hidden = model.encoder(english_tensor, eng_len)
        x = torch.tensor([[BOS_IDX]], dtype=torch.long).to(device)
        predicted_words = []
        for _ in range(max_len):
            prediction, hidden = model.decoder(x, hidden)
            top1 = prediction.argmax(1).item()
            if top1 == EOS_IDX:
                break
            predicted_words.append(top1)
            x = torch.tensor([[top1]], dtype=torch.long).to(device)
    return predicted_words


## 5. 执行训练

In [12]:
EPOCHS = 100
PATIENCE = 5
best_val_loss = float("inf")
early_stop_counter = 0

print("开始训练...")
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss = evaluate(model, val_loader, criterion, DEVICE)

    print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        early_stop_counter += 1
        if early_stop_counter >= PATIENCE:
            print(f"连续 {PATIENCE} 轮验证集 Loss 未下降，触发 Early Stopping!")
            break

# 恢复最优模型
model.load_state_dict(torch.load("best_model.pt"))
print("已加载验证集最优模型进行测试。")

for k in range(10):
  print("\n======== 翻译效果演示 ========")
  sample_en, sample_zh = test_data[k]
  print("原始英文:", sample_en)
  print("目标中文:", sample_zh)

  sample_en_idx = [en_vocab.get(w, UNK_IDX) for w in sample_en.split()]
  pred_idx = translate(model, sample_en_idx, DEVICE)
  pred_zh = " ".join([zh_idx2word.get(i, "<UNK>") for i in pred_idx])
  print("模型预测:", pred_zh)


开始训练...
Epoch [1/100], Train Loss: 3.4268, Val Loss: 2.7470
Epoch [2/100], Train Loss: 2.2808, Val Loss: 1.8900
Epoch [3/100], Train Loss: 1.6227, Val Loss: 1.4798
Epoch [4/100], Train Loss: 1.3150, Val Loss: 1.2230
Epoch [5/100], Train Loss: 1.1645, Val Loss: 1.1508
Epoch [6/100], Train Loss: 1.1149, Val Loss: 1.1147
Epoch [7/100], Train Loss: 1.0870, Val Loss: 1.0856
Epoch [8/100], Train Loss: 1.0571, Val Loss: 1.0499
Epoch [9/100], Train Loss: 1.0147, Val Loss: 1.0040
Epoch [10/100], Train Loss: 0.9558, Val Loss: 0.9392
Epoch [11/100], Train Loss: 0.8888, Val Loss: 0.8715
Epoch [12/100], Train Loss: 0.8267, Val Loss: 0.8119
Epoch [13/100], Train Loss: 0.7815, Val Loss: 0.7743
Epoch [14/100], Train Loss: 0.7486, Val Loss: 0.7425
Epoch [15/100], Train Loss: 0.7226, Val Loss: 0.7191
Epoch [16/100], Train Loss: 0.6985, Val Loss: 0.6950
Epoch [17/100], Train Loss: 0.6769, Val Loss: 0.6746
Epoch [18/100], Train Loss: 0.6554, Val Loss: 0.6526
Epoch [19/100], Train Loss: 0.6329, Val Loss: 0